# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sohamrathi17/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

My lane is **Refresh / Content Opportunity Scoring** (ML-02, ML-03). The contract, in five plain answers:

1. **One row means:** one page (`content_hash_id`, within its `client_hash_id`) at one decision moment, **31 March 2026**. The raw daily table is one row per page per day; I collapse March's daily rows into one row per page.
2. **Tables I use:** `fact_content_daily_performance` (partitions `month=2026-03` for features and `month=2026-04` for the label) and `dim_content` (only `content_created_date`, for page age).
3. **Time window:** features come from **1-31 March 2026** only. The label comes from **1-30 April 2026**. Nothing from April touches a feature. This is a mid-panel month on purpose: June 2026 (the `_sample` table) stays sealed as a future test month.
4. **What I predict (label):** `is_declining` = 1 when the page's average daily impressions in April are more than 20% below its average daily impressions in March. Observed outcome from Search Console, one month *after* the decision moment, so it is a real forecast, not a description of a trend that already happened.
5. **What I deliberately exclude:** `content_updated_date` and `last_optimized_date` from `dim_content`. That table is a snapshot taken after the panel ended, so these dates can be *after* 31 March: for most pages in my slice the "last updated" date is in the future relative to the decision moment (measured in the feature cell below). Using them would leak the future in, and would also partly encode the action (a refresh) I am trying to recommend.

**Slice rules:** a page enters the slice only if it has GSC data on every day of March (31 rows with `gsc_data_available IS TRUE`) and of April (30 rows), and at least 100 impressions in March, so a ±20% change means something.

In [1]:
import os, sys, subprocess, getpass

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

# Token order: env var -> Colab Secret -> local `hf auth login` cache -> prompt (last resort).
# Never type the token into a cell: this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    from huggingface_hub import get_token
    HF_TOKEN = get_token()
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download

# Download each file once into the Hugging Face cache (outside the repo, never committed).
# Re-runs read the local copy, so no repeated remote scans and no 429s.
def hf_file(path):
    return hf_hub_download("FlyRank/internship-warehouse", path, repo_type="dataset", token=HF_TOKEN)

MAR = f"read_parquet('{hf_file('fact_content_daily_performance/month=2026-03/data_0.parquet')}')"
APR = f"read_parquet('{hf_file('fact_content_daily_performance/month=2026-04/data_0.parquet')}')"
DIM_CONTENT = f"read_parquet('{hf_file('dim_content.parquet')}')"

con = duckdb.connect()
print("Tables ready: March + April daily partitions, dim_content")

Tables ready: March + April daily partitions, dim_content


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | the five built in section 3 from March daily rows + `content_created_date` | all knowable on 31 March |
| **Label / proxy** | `is_declining` (from April `gsc_impressions`) | the outcome I predict, one month after the decision moment |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date`, `gsc_data_available` | grouping, joining, client-holdout splits, filtering. IDs are never features |
| **Excluded** | `content_updated_date`, `last_optimized_date` | snapshot dates that can be after 31 March (future leak) |
| **Excluded** | every April column as a feature | that is the label window |
| **Excluded** | GA4 columns (sessions, engagement) for now | `ga4_data_available` is mostly FALSE or NULL in March (checked in query 3), so they would be missing for most pages; zeros there mean "not tracked", not "no engagement" |

## 3. Verify it with queries (grain, counts, missing values, windows)

Three queries on the March 2026 partition, each checking one claim from the contract.

**Query 1: grain.** Claim: the daily table has exactly one row per `report_date × client × page`. If any combination appears twice, this returns it.

In [2]:
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {MAR}
    GROUP BY ALL
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Duplicate (report_date, client, page) rows found: {len(dupes)}")
print("Grain holds: one row per page per day." if dupes.empty else dupes)

Duplicate (report_date, client, page) rows found: 0
Grain holds: one row per page per day.


**Query 2: row count and date span.** Claim: the March partition covers 1-31 March 2026 only, and tells me how big the raw slice is before my filters.

In [3]:
con.sql(f"""
    SELECT COUNT(*)                         AS daily_rows,
           MIN(report_date)                 AS first_day,
           MAX(report_date)                 AS last_day,
           COUNT(DISTINCT report_date)      AS days,
           COUNT(DISTINCT client_hash_id)   AS clients,
           COUNT(DISTINCT content_hash_id)  AS pages
    FROM {MAR}
""").df()

,daily_rows,first_day,last_day,days,clients,pages
0,9841378,2026-03-01,2026-03-31,31,55,331437


**Query 3: availability, filtered with `IS TRUE`.** The flags are three-valued (TRUE / FALSE / NULL), so `= TRUE` or `NOT ...` can silently mishandle NULLs. I count every state of both flags, then show how many rows survive `gsc_data_available IS TRUE`, the filter my features use.

In [4]:
con.sql(f"""
    SELECT 'gsc_data_available' AS flag,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS is_true,
           COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS is_false,
           COUNT(*) FILTER (WHERE gsc_data_available IS NULL)  AS is_null,
           ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) AS pct_surviving,
           COUNT(DISTINCT client_hash_id) FILTER (WHERE gsc_data_available IS TRUE) AS clients_surviving,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS NOT TRUE) AS impressions_outside_filter
    FROM {MAR}
    UNION ALL
    SELECT 'ga4_data_available',
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE),
           COUNT(*) FILTER (WHERE ga4_data_available IS FALSE),
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL),
           ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1),
           COUNT(DISTINCT client_hash_id) FILTER (WHERE ga4_data_available IS TRUE),
           NULL
    FROM {MAR}
""").df()

,flag,is_true,is_false,is_null,pct_surviving,clients_surviving,impressions_outside_filter
0,gsc_data_available,3611061,6230317,0,36.7,47,0.0
1,ga4_data_available,413966,6408671,3018741,4.2,41,NaN


Only about a third of March's daily rows survive the GSC filter, and the rows outside it carry zero impressions, so nothing is lost by dropping them. GA4 is available on only a few percent of rows, which is why GA4 fields stay out for now.

### Five features (max), built from March only

| Feature | What it is | Knowable at the decision moment because... |
|---|---|---|
| `log_imp_mar` | log of total March impressions | it is summed from March 1-31 rows, all reported by 31 March |
| `avg_position_mar` | impression-weighted average position in March (0 = no data, skipped) | it is computed from March rows only |
| `ctr_mar` | March clicks / March impressions (a fraction, not ×100) | March clicks and impressions are both known by 31 March |
| `age_days` | days from `content_created_date` to 31 March | a creation date does not change later, and every page in the slice was created before 31 March (checked below) |
| `momentum_mar` | daily impressions in 17-31 March / daily impressions in 1-16 March | both halves are inside March, so it measures direction *within* the feature window |

In [5]:
features = con.sql(f"""
    WITH mar AS (
        SELECT client_hash_id, content_hash_id,
               COUNT(*)              AS gsc_days_mar,
               SUM(gsc_impressions)  AS imp_mar,
               SUM(gsc_clicks)       AS clk_mar,
               SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_sum_position END)::DOUBLE
                 / NULLIF(SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_impressions END), 0) AS avg_position_mar,
               SUM(CASE WHEN report_date >= DATE '2026-03-17' THEN gsc_impressions END) / 15.0 AS imp_late_per_day,
               SUM(CASE WHEN report_date <  DATE '2026-03-17' THEN gsc_impressions END) / 16.0 AS imp_early_per_day
        FROM {MAR}
        WHERE gsc_data_available IS TRUE
        GROUP BY ALL
        HAVING COUNT(*) = 31 AND SUM(gsc_impressions) >= 100
    ),
    apr AS (   -- label window only: never joined in as a feature
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_apr
        FROM {APR}
        WHERE gsc_data_available IS TRUE
        GROUP BY ALL
        HAVING COUNT(*) = 30
    )
    SELECT m.*, a.imp_apr, d.content_created_date, d.content_updated_date
    FROM mar m
    JOIN apr a USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
""").df()

DECISION = pd.Timestamp("2026-03-31")
features["log_imp_mar"]  = np.log1p(features["imp_mar"])
features["ctr_mar"]      = features["clk_mar"] / features["imp_mar"]
features["age_days"]     = (DECISION - pd.to_datetime(features["content_created_date"])).dt.days
features["momentum_mar"] = (features["imp_late_per_day"] + 1) / (features["imp_early_per_day"] + 1)

# Label: April daily impressions more than 20% below March daily impressions
features["is_declining"] = ((features["imp_apr"] / 30) < 0.8 * (features["imp_mar"] / 31)).astype(int)

FEATURES = ["log_imp_mar", "avg_position_mar", "ctr_mar", "age_days", "momentum_mar"]
assert features["content_hash_id"].is_unique, "grain broke: more than one row per page"
print(f"Feature frame: {len(features):,} pages (one row each), {features['client_hash_id'].nunique()} clients")
print(f"Declining in April: {features['is_declining'].mean():.1%}")
print(f"Pages created after the decision moment: {(features['age_days'] < 0).sum()}  (must be 0)")
print(f"Missing values in the five features: {int(features[FEATURES].isna().sum().sum())}")
upd_after = (pd.to_datetime(features["content_updated_date"]) > DECISION).mean()
print(f"Why content_updated_date is excluded: {upd_after:.1%} of these pages show a 'last updated' date AFTER 31 March")
features[FEATURES + ["is_declining"]].describe().round(3)

Feature frame: 50,299 pages (one row each), 32 clients
Declining in April: 47.4%
Pages created after the decision moment: 0  (must be 0)
Missing values in the five features: 0
Why content_updated_date is excluded: 77.9% of these pages show a 'last updated' date AFTER 31 March


,log_imp_mar,avg_position_mar,ctr_mar,age_days,momentum_mar,is_declining
count,50299.000,50299.000,50299.000,50299.000,50299.000,50299.000
mean,7.609,12.516,0.003,197.359,1.174,0.474
std,1.229,11.977,0.004,110.112,0.922,0.499
min,4.615,0.227,0.000,31.000,0.043,0.000
25%,6.685,4.510,0.000,85.000,0.745,0.000
50%,7.558,7.446,0.002,194.000,0.996,0.000
75%,8.495,16.953,0.004,263.000,1.320,1.000
max,12.409,87.711,0.135,494.000,52.925,1.000


### The trap: one label-derived column, on purpose

Quick score: a gradient-boosted tree, trained on some clients and tested on **clients it never saw** (client-holdout split, seed 42), scored by ROC-AUC (0.5 = coin flip, 1.0 = perfect).

First with my five honest features. Then I add `imp_apr_per_day`, April's average daily impressions. The label is computed *from* that number, so it is the answer in disguise.

In [6]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

features["imp_apr_per_day"] = features["imp_apr"] / 30   # LEAK, on purpose

train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
                           .split(features, groups=features["client_hash_id"]))
train, test = features.iloc[train_idx], features.iloc[test_idx]
print(f"train: {len(train):,} pages / {train['client_hash_id'].nunique()} clients | "
      f"test: {len(test):,} pages / {test['client_hash_id'].nunique()} clients | test declining rate {test['is_declining'].mean():.1%}\n")

def quick_score(cols):
    model = HistGradientBoostingClassifier(random_state=42).fit(train[cols], train["is_declining"])
    return roc_auc_score(test["is_declining"], model.predict_proba(test[cols])[:, 1])

leaky_auc = quick_score(FEATURES + ["imp_apr_per_day"])
print(f"WITH the leak    (5 features + imp_apr_per_day): ROC-AUC {leaky_auc:.3f}  <- looks near perfect")

train: 11,617 pages / 24 clients | test: 38,682 pages / 8 clients | test declining rate 51.0%



WITH the leak    (5 features + imp_apr_per_day): ROC-AUC 0.998  <- looks near perfect


Near perfect, and it means nothing: the model just learned to recompute the label from April numbers, which nobody has on 31 March. Now I delete the leaked column and keep the honest number.

In [7]:
features = features.drop(columns=["imp_apr_per_day"])
train, test = features.iloc[train_idx], features.iloc[test_idx]

honest_auc = quick_score(FEATURES)
print(f"WITHOUT the leak (5 honest features):            ROC-AUC {honest_auc:.3f}  <- the number I keep")
print("Leaked column still present:", "imp_apr_per_day" in features.columns)

WITHOUT the leak (5 honest features):            ROC-AUC 0.616  <- the number I keep
Leaked column still present: False


The honest number is much lower, and that is the real starting point: five March-only features give a modest but real signal about April, on clients the model has never seen. Improving that honestly (better features, the baseline rule comparison) is the work of the next weeks.

## 4. Data limits

**Named limitation: my slice is dominated by a few clients.** The pages that pass the slice rules are not spread evenly across clients (measured below). A single large client can hold a big share of all pages, and a client-holdout split can then put most of the test set in one client. So the quick score above mostly reflects how well the model does on *that* client's site. For the capstone I will report results per client and use grouped cross-validation over several folds, not one split.

Other limits I know about:
- **Coverage filter:** requiring GSC data on all 31 days of March and 30 days of April drops pages that were registered mid-month or that lost tracking. Pages that vanish in April are excluded, so the slice slightly favours pages that kept being tracked.
- **One month of features, one month of outcome.** Seasonality (e.g. an April holiday dip) can look like decline. A single month-to-month change cannot separate the two.
- **Observational data.** Even with a clean future label, this can say which pages tend to decline, never *why* or what a refresh would cause.

In [8]:
pages_per_client = features.groupby("client_hash_id").size().sort_values(ascending=False)
share = pages_per_client / pages_per_client.sum()
print(f"Clients in the slice: {len(pages_per_client)}")
print(f"Largest client holds {share.iloc[0]:.1%} of all pages; top 3 hold {share.iloc[:3].sum():.1%}")
print(f"Median pages per client: {pages_per_client.median():.0f}")
print(f"Test-set share from the largest test client: "
      f"{test.groupby('client_hash_id').size().max() / len(test):.1%}")

Clients in the slice: 32
Largest client holds 29.0% of all pages; top 3 hold 66.6%
Median pages per client: 198
Test-set share from the largest test client: 37.7%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.